# 05. Inventory Supply and Quality

Venture's recommendation system can only be as strong as the listings available to it. This notebook examines public supply, verification, category balance, neighbourhood coverage, and upcoming events.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent

sys.path.insert(0, str(WORKSPACE / "src"))

from venture_ds import (
    load_exports,
    load_optional_export,
    safe_rate,
    wilson_interval,
    ordered_session_funnel,
    add_sample_warning,
)

exports = load_exports(WORKSPACE / "data" / "raw")
events = exports["events"]
sessions = exports["sessions"]
feedback = exports["feedback"]
inventory = exports["inventory"]

exports["_files"]

,dataset,path,found
0,events,C:\Users\ppmal\Documents\Projects\venture-prod...,True
1,sessions,C:\Users\ppmal\Documents\Projects\venture-prod...,True
2,feedback,C:\Users\ppmal\Documents\Projects\venture-prod...,True
3,inventory,C:\Users\ppmal\Documents\Projects\venture-prod...,True


In [2]:
if inventory.empty:
    print("No inventory export found.")
else:
    inventory_summary = {
        "total": len(inventory),
        "published": inventory["status"].eq("published").sum(),
        "trusted": inventory["verification_status"].isin(
            ["official_source", "verified_venture", "verified_business"]
        ).sum(),
        "places": inventory["entity_type"].eq("place").sum(),
        "events": inventory["entity_type"].eq("event").sum(),
        "missing_neighbourhood": inventory["neighbourhood"].isna().sum(),
    }
    display(pd.Series(inventory_summary))

total                    28
published                10
trusted                   6
places                   27
events                    1
missing_neighbourhood    22
dtype: int64

In [3]:
if not inventory.empty:
    category_table = (
        inventory.groupby(["category", "status"])
        .size()
        .unstack(fill_value=0)
        .sort_values("published" if "published" in inventory["status"].unique() else inventory["status"].iloc[0], ascending=False)
    )
    display(category_table.head(20))

    neighbourhood_table = (
        inventory.groupby("neighbourhood", dropna=False)
        .agg(
            listings=("entity_id", "size"),
            published=("status", lambda values: values.eq("published").sum()),
            trusted=("verification_status", lambda values: values.isin(
                ["official_source", "verified_venture", "verified_business"]
            ).sum()),
        )
        .sort_values(["published", "trusted"], ascending=False)
    )
    display(neighbourhood_table.head(20))

status,draft,published
category,,
Anime Convention,0,1
Bakery,1,1
Bar And Grill Restaurant,0,1
Pizza Restaurant,5,1
Brunch,0,1
Fine dining,0,1
Convention Centre,0,1
Mexican,0,1
Steakhouse,0,1


,listings,published,trusted
neighbourhood,,,
NaN,22,4,0
Downtown,2,2,2
Exchange District,2,2,2
South Osborne,1,1,1
The Forks,1,1,1


In [4]:
if not inventory.empty and "start_at" in inventory.columns:
    upcoming_events = inventory[
        inventory["entity_type"].eq("event")
        & inventory["start_at"].notna()
        & (inventory["start_at"] >= pd.Timestamp.now(tz="UTC"))
    ].copy()

    print(f"Upcoming events in export: {len(upcoming_events)}")
    display(
        upcoming_events.sort_values("start_at")
        [["name", "category", "neighbourhood", "start_at"]]
        .head(20)
    )

Upcoming events in export: 0


,name,category,neighbourhood,start_at
